In [1]:
import os
%pwd
os.chdir('../')

In [2]:
import pandas as pd

data=pd.read_csv('artifacts/data_ingestion/winequality-red.csv')
data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [4]:
data.columns

Index(['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates', 'alcohol', 'quality'],
      dtype='object')

In [5]:
data.isnull().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [6]:
data.shape

(1599, 12)

In [7]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir:Path
    STATUS_FILE:str
    unzip_data_dir:Path
    all_schema: dict

In [8]:
from WineQuality.constants import *
from WineQuality.utils.common import read_yaml,create_directories

class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):

                 self.config=read_yaml(config_filepath)
                 self.params=read_yaml(params_filepath)
                 self.schema=read_yaml(schema_filepath)                 
                 create_directories([self.config.artifacts_root])

    def get_data_validation_config(self)->DataValidationConfig:
            config=self.config.data_validation
            schema=self.schema.COLUMNS

            create_directories([config.root_dir])
            data_validation_config=DataValidationConfig(
                    root_dir=config.root_dir,
                    STATUS_FILE=config.STATUS_FILE,
                    all_schema=schema,
                    unzip_data_dir=config.unzip_data_dir
            )                 
                    
            return data_validation_config

In [ ]:
import os 
import urllib.request as request
import zipfile
from WineQuality import logger
from WineQuality.utils.common import get_size

class DataValidation:
    def __init__(self,config:DataValidationConfig):
        self.config=config

    def validate_all_columns(self) -> bool:
        try:
            validation_status = True

            data = pd.read_csv(self.config.unzip_data_dir)

            dataset_cols = set(data.columns)
            schema_cols = set(self.config.all_schema.keys())

            # 1. Missing columns
            missing_cols = schema_cols - dataset_cols

            # 2. Extra columns
            extra_cols = dataset_cols - schema_cols

            # 3. Empty dataset
            if data.empty:
                validation_status = False
                logger.error("Dataset is empty")

            # 4. Missing columns check
            if missing_cols:
                validation_status = False
                logger.error(f"Missing columns: {missing_cols}")

            # 5. Unexpected columns check
            if extra_cols:
                validation_status = False
                logger.error(f"Extra columns: {extra_cols}")

            # 6. Null value check
            null_cols = data.columns[data.isnull().any()].tolist()
            if null_cols:
                logger.warning(f"Columns containing null values: {null_cols}")

            # 7. Duplicate row check
            duplicate_rows = data.duplicated().sum()
            if duplicate_rows > 0:
                logger.warning(f"Duplicate rows found: {duplicate_rows}")

            # 8. Data type validation
            for col, expected_dtype in self.config.all_schema.items():
                if col in data.columns:
                    actual_dtype = str(data[col].dtype)

                    if expected_dtype not in actual_dtype:
                        validation_status = False
                        logger.error(
                            f"Datatype mismatch in {col}. "
                            f"Expected: {expected_dtype}, Found: {actual_dtype}"
                        )

            with open(self.config.STATUS_FILE, "w") as f:
                f.write(f"Validation status: {validation_status}")

            return validation_status

        except Exception as e:
            logger.error(e)
            raise e
                
            

In [10]:
try:
    config=ConfigurationManager()
    data_validation_config=config.get_data_validation_config()
    data_validation=DataValidation(config=data_validation_config)
    data_validation.validate_all_columns()
except Exception as e:
    raise e

FILE: config\config.yaml
TYPE: <class 'dict'>
CONTENT: {'artifacts_root': 'artifacts', 'data_ingestion': {'root_dir': 'artifacts/data_ingestion', 'source_URL': 'https://github.com/entbappy/Branching-tutorial/raw/master/winequality.zip', 'local_data_file': 'artifacts/data_ingestion/data.zip', 'unzip_dir': 'artifacts/data_ingestion'}, 'data_validation': {'root_dir': 'artifacts/data_validation', 'unzip_data_dir': 'artifacts/data_ingestion/winequality-red.csv', 'STATUS_FILE': 'artifacts/data_validation/status.txt'}}
[2026-06-13 09:27:36,002:INFO:common:yaml file: config\config.yaml loaded successfully]
FILE: params.yaml
TYPE: <class 'dict'>
CONTENT: {'key': 'val'}
[2026-06-13 09:27:36,005:INFO:common:yaml file: params.yaml loaded successfully]
FILE: schema.yaml
TYPE: <class 'dict'>
CONTENT: {'COLUMNS': {'fixed acidity': 'float64', 'volatile acidity': 'float64', 'citric acid': 'float64', 'residual sugar': 'float64', 'chlorides': 'float64', 'free sulfur dioxide': 'float64', 'total sulfur dio